# Step 6 - Clean coastdown data
> From the coasting data (from all laps), remove anything below 5 m/s and anything below a second.

Starts from Step 5's coasting (no throttle, no brake, moving), inside the Step 4 lap windows.

In [ ]:
import polars as pL
import numpy as np
import matplotlib.pyplot as plt

pd = pL.read_parquet("../software-data.parquet")
pd = pd.fill_null(strategy="forward")

rpm = pd["SME_TRQSPD_Speed"]
time = pd["Time"]

gear_ratio = 12/41
radius = .2                             #in meters

def meters_second(rpm):
    speed_mps = (rpm * gear_ratio * 2 * 3.14159 * radius) / 60  # Convert rpm to m/s
    return speed_mps

speed = meters_second(rpm)
moving = speed > 0.1

In [ ]:
def find_periods(condition):
    # turns a True/False mask into lists of start and end times
    # merges gaps under 0.1 s, drops periods 0.7 s or shorter
    total_periods = time.filter(condition)
    gaps = total_periods.diff()
    starts = total_periods.filter(gaps > .1)
    ends = total_periods.shift(1).filter(gaps > .1)

    starts = starts.to_list()
    ends = ends.to_list()
    total_periods = total_periods.to_list()
    starts = [total_periods[0]] + starts
    ends = ends + [total_periods[-1]]

    filtered_starts = []
    filtered_ends = []
    for s, e in zip(starts, ends):
        if e - s > 0.7:
            filtered_starts.append(s)
            filtered_ends.append(e)

    return filtered_starts, filtered_ends

In [ ]:
# lap windows from Step 4
lap_mask = (time >= 20) & (time <= 68)
lap_mask2 = (time >= 95) & (time <= 133)

brake = pd["ETC_STATUS_BRAKE_SENSE_VOLTAGE"]
pedal = pd["ETC_STATUS_PEDAL_TRAVEL"]

# Step 5 coasting: no throttle, no brake, still moving
no_throttle = pedal == 0
no_brake = brake <= 375
coasting_5 = no_throttle & no_brake & moving

## Order of the two filters
Remove readings below 5 m/s **first**, then drop any piece that is now shorter than 1 s. Whatever is left has to pass both rules.

Why it matters: a coast can be long overall but mostly below 5 m/s. If a 4 s coast has only 0.5 s above 5 m/s, checking the length first would pass it (4 s), and the speed cut would then leave a 0.5 s piece that slips through.

In [ ]:
# Step 6: coasting inside the laps, above 5 m/s, at least 1 s long
# speed cut first, then the 1 s check on what's left
in_laps = lap_mask | lap_mask2
fast_enough = speed >= 5
coast_6 = coasting_5 & in_laps & fast_enough

c6_starts, c6_ends = find_periods(coast_6)

clean_starts = []
clean_ends = []
for s, e in zip(c6_starts, c6_ends):
    if e - s >= 1:
        clean_starts.append(s)
        clean_ends.append(e)

# keep each coast as its own time + speed segment (Step 7 needs dv/dt inside each one)
coast_segments = []
for s, e in zip(clean_starts, clean_ends):
    seg_mask = (time >= s) & (time <= e) & fast_enough   # also drop the few dips under 5 m/s
    seg_time = time.filter(seg_mask)
    seg_speed = speed.filter(seg_mask)
    coast_segments.append((seg_time, seg_speed))

print("Clean coasts:")
for seg_time, seg_speed in coast_segments:
    print(f"   {seg_time[0]:.2f} - {seg_time[-1]:.2f}  ({seg_time[-1] - seg_time[0]:.2f} s)  "
          f"speed {seg_speed.max():.2f} -> {seg_speed.min():.2f} m/s  ({len(seg_speed)} rows)")

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(time, speed, color='gray', label="speed")
for seg_time, seg_speed in coast_segments:
    plt.plot(seg_time, seg_speed, color='blue', linewidth=3)
plt.plot([], [], color='blue', linewidth=3, label="clean coast data")   # one legend entry
plt.axhline(5, color='red', linestyle='--', label="5 m/s")
plt.xlim(15, 140)
plt.xlabel("Time (s)")
plt.ylabel("Speed (m/s)")
plt.title("Clean Coastdown Data")
plt.legend()
plt.grid(True)
plt.show()

## Result: 4 clean coasts, 38.89 s, 3,259 rows

| Lap | Coast | Length | Speed |
|---|---|---|---|
| 1 | 24.63 - 38.77 s | 14.14 s | 10.54 -> 5.00 m/s |
| 1 | 51.80 - 61.22 s | 9.43 s | 8.20 -> 5.00 m/s |
| 2 | 99.33 - 103.03 s | 3.70 s | 10.70 -> 9.02 m/s |
| 2 | 103.83 - 115.46 s | 11.62 s | 8.81 -> 5.01 m/s |

**Removed:** Lap 1's 42.88 - 46.41 s and 48.63 - 49.53 s coasts (never reach 5 m/s), and the slow ends of three others (41.18 -> 38.77 s, 68.00 -> 61.22 s, 132.96 -> 115.46 s).

Notes:
- Near the end of a coast the speed hovers right at 5 m/s and a few readings dip just under. Those rows are removed too, leaving small holes (largest 0.083 s).
- The 1 s check removed nothing here: `find_periods` already drops anything 0.7 s or shorter, and no piece fell between 0.7 and 1 s.
- Each coast is kept as its own segment so Step 7 doesn't compute dv/dt across the jump from one coast to the next.